In [5]:
import requests
import re
import pandas as pd
import json
import time
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
os.environ["API_KEY"] = api_key
url = "https://api-v1.zyrooai.com/api/v1/math-classifier/interview/questions"

response = requests.get(url)
data = response.json()
df = pd.DataFrame(data['data'])

print(f"Data Loaded: {len(df)} questions found.")
df.head(10)

Data Loaded: 67 questions found.


,id,grade,question,answer,section,label,options
0,P3-002,P3,What is the remainder when 3079 is divided by 8?,7,Section B,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...",NaN
1,P3-006,P3,Ethan left his home at 10:45 am and took 25 mi...,(a) 35 min\n(b) 1:10 pm,Section C,"{'strand': 'MEASUREMENT AND GEOMETRY', 'subStr...",NaN
2,P3-012,P3,5 x 6 = _____,(2) 5 + 5 + 5 + 5 + 5 + 5,Section A,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...","[5 x 5 x 5 x 5 x 5 x5, 5 + 5 + 5 + 5 + 5 + 5, ..."
3,P3-013,P3,Mr Tan bought 6 loaves of bread. Each loaf is ...,(3) 48,Section A,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...","[2, 14, 48, 68]"
4,P3-014,P3,Mr Aw had $650. He had $127 more than Mr Goh. ...,523,Section B,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...",NaN
5,P3-015,P3,Mrs Siva has 650 sweets. She packed all the sw...,2,Section B,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...",NaN
6,P3-016,P3,"4 thousands, 6 hundreds and 9 ones is the same...",(2) 4609,Section A,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...","[4069, 4609, 4690, 4960]"
7,P3-017,P3,What is the missing number in the following nu...,(3) 2425,Section A,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...","[2405, 2415, 2425, 2435]"
8,P3-018,P3,"I am a number between 20 and 40, You can find ...",(1) 24,Section A,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...","[24, 28, 30, 32]"
9,P3-019,P3,Write 9544 in words.,nine thousand five hundred and forty-four,Section B,"{'strand': 'NUMBER AND ALGEBRA', 'subStrand': ...",NaN


In [ ]:
syllabus_df = pd.read_csv('Syllabus.csv')
syllabus_df[['strand', 'subStrand', 'topic']] = syllabus_df[['strand', 'subStrand', 'topic']].ffill()
def clean_topic(text):
    if pd.isna(text):
        return text
    # This regex looks for digits and a dot at the start: "^[0-9]+\.\s*"
    return re.sub(r'^[0-9]+\.\s*', '', str(text))
syllabus_df['topic'] = syllabus_df['topic'].apply(clean_topic)

syllabus_df['loId'] = (
    syllabus_df['grade'].astype(str) + ":" + 
    syllabus_df['subStrand'].astype(str) + ":" + 
    syllabus_df['ref'].astype(str)
)
syllabus_df.head(5)

,strand,subStrand,topic,ref,learningOutcome,grade,loId
0,NUMBER AND ALGEBRA,WHOLE NUMBERS,Numbers up to 10 000,1.1.1,counting in hundreds/thousands,P3,P3:WHOLE NUMBERS:1.1.1
1,NUMBER AND ALGEBRA,WHOLE NUMBERS,Numbers up to 10 000,1.1.2,"number notation, representations and place val...",P3,P3:WHOLE NUMBERS:1.1.2
2,NUMBER AND ALGEBRA,WHOLE NUMBERS,Numbers up to 10 000,1.1.3,reading and writing numbers in numerals and in...,P3,P3:WHOLE NUMBERS:1.1.3
3,NUMBER AND ALGEBRA,WHOLE NUMBERS,Numbers up to 10 000,1.1.4,comparing and ordering numbers,P3,P3:WHOLE NUMBERS:1.1.4
4,NUMBER AND ALGEBRA,WHOLE NUMBERS,Numbers up to 10 000,1.1.5,patterns in number sequences,P3,P3:WHOLE NUMBERS:1.1.5


In [4]:
syllabus_knowledge_base = ""
for _, row in syllabus_df.iterrows():
    loId = f"{row['grade']}:{row['subStrand']}:{row['ref']}"
    syllabus_knowledge_base += (
        f"loId: {loId} | Strand: {row['strand']} | SubStrand: {row['subStrand']} | "
        f"Topic: {row['topic']} | Outcome: {row['learningOutcome']}\n"
    )
syllabus_knowledge_base

'loId: P3:WHOLE NUMBERS:1.1.1 | Strand: NUMBER AND ALGEBRA | SubStrand: WHOLE NUMBERS | Topic: Numbers up to 10 000 | Outcome: counting in hundreds/thousands\nloId: P3:WHOLE NUMBERS:1.1.2 | Strand: NUMBER AND ALGEBRA | SubStrand: WHOLE NUMBERS | Topic: Numbers up to 10 000 | Outcome: number notation, representations and place values (thousands, hundreds, tens, ones)\nloId: P3:WHOLE NUMBERS:1.1.3 | Strand: NUMBER AND ALGEBRA | SubStrand: WHOLE NUMBERS | Topic: Numbers up to 10 000 | Outcome: reading and writing numbers in numerals and in words\nloId: P3:WHOLE NUMBERS:1.1.4 | Strand: NUMBER AND ALGEBRA | SubStrand: WHOLE NUMBERS | Topic: Numbers up to 10 000 | Outcome: comparing and ordering numbers\nloId: P3:WHOLE NUMBERS:1.1.5 | Strand: NUMBER AND ALGEBRA | SubStrand: WHOLE NUMBERS | Topic: Numbers up to 10 000 | Outcome: patterns in number sequences\nloId: P3:WHOLE NUMBERS:1.2.1 | Strand: NUMBER AND ALGEBRA | SubStrand: WHOLE NUMBERS | Topic: Addition and Subtraction | Outcome: additi

In [36]:
# Configure Gemini
client = genai.Client(api_key=os.environ["API_KEY"])

def classification_engine(question_text):
    prompt = f"""    
    ### TASK:
    Find the EXACT matching entry from the SYLLABUS KNOWLEDGE BASE for the question provided.
    
    ### RULES:
    1. You MUST use one of the loIds provided in the list below.
    2. The 'strand', 'subStrand', 'topic', and 'learningOutcome' MUST be copied word-for-word from that specific loId entry.
    3. If multiple outcomes apply, pick the most specific one.

    ### SYLLABUS KNOWLEDGE BASE:
    {syllabus_knowledge_base}

    ### QUESTION TO CLASSIFY:
    "{question_text}"

    ### OUTPUT FORMAT (JSON ONLY):
    {{
      "strand": "...",
      "subStrand": "...",
      "topic": "...",
      "learningOutcome": "...",
      "loId": "..."
    }}
    """
    
    try:
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=prompt,
        )
        clean_json = response.text.replace('```json', '').replace('```', '').strip()
        return json.loads(clean_json)
    except Exception as e:
        return {"error": str(e)}

# Test the engine once
print(classification_engine("Ethan left his home at 10:45 am and took 25 minutes to cycle to the hawker centre. After having his lunch, he left the hawker centre at 11:45 am. \n(a) How long was Ethan at the hawker centre?\n(b) From the hawker centre, Ethan cycled for 10 minutes to the supermarket. He spent 1 h 15 min in the supermarket before cycling home. What time did Ethan leave the supermarket?"))

{'strand': 'MEASUREMENT AND GEOMETRY', 'subStrand': 'MEASUREMENT', 'topic': 'Time', 'learningOutcome': 'finding the starting time, finishing time or duration given the other two quantities', 'loId': 'P3:MEASUREMENT:4.2.2'}


In [42]:
output_file = "classification_results.jsonl"
print("Starting classification...")

# 1. Check existing progress (Standard Resume Logic)
already_processed_count = 0
if os.path.exists(output_file):
    with open(output_file, 'r', encoding="utf-8") as f:
        already_processed_count = sum(1 for line in f)
print(f"Resuming from question {already_processed_count}...")

# 2. Open file in append mode
with open(output_file, "a", encoding="utf-8") as f:
    for index, row in df.iterrows():
        # Skip rows that are already saved in the file
        if index < already_processed_count:
            continue
            
        if index % 5 == 0: 
            print(f"Processing: {index}/{len(df)}")
        
        # Get prediction
        prediction = classification_engine(row['question'])
        
        # 3. MODIFIED ERROR HANDLING: Stop on error
        if not isinstance(prediction, dict) or "error" in prediction:
            error_msg = prediction.get('error', 'Unknown error')
            print(f"\n🛑 STOPPING: Error at index {index}: {error_msg}")
            print("You can run this cell again to resume from this exact point.")
            break  # This exits the loop and DOES NOT save the record
            
        # Construct the final record
        record = {
            "original_index": index,
            "question": row['question'],
            "expected_loId": row['label']['loId'], 
            **prediction 
        }
        
        # 4. Save to file
        f.write(json.dumps(record) + "\n")
        f.flush() 
        
        # API Rate Limit Safety
        time.sleep(4) 

# Only load final_df if the loop actually finished all questions
results = []
if os.path.exists(output_file):
    with open(output_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue # Skip empty lines
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"⚠️ Skipping corrupted data at line {line_num}")

    final_df = pd.DataFrame(results)
    
    if len(final_df) == len(df):
        print(f"\n✅ Finished! All {len(df)} results processed.")
    else:
        print(f"\n⚠️ Process paused. {len(final_df)}/{len(df)} questions completed.")

Starting classification...
Resuming from question 67...

✅ Finished! All 67 results processed.


In [ ]:
syllabus_lookup = syllabus_df.set_index('loId').to_dict('index')

# Function to safely get expected values
def get_expected_info(loId, field):
    if loId in syllabus_lookup:
        return syllabus_lookup[loId].get(field, "N/A")
    return "N/A"

# Create the missing 'expected_' columns in your results dataframe
for level in ['strand', 'subStrand', 'topic', 'learningOutcome']:
    final_df[f'expected_{level}'] = final_df['expected_loId'].apply(lambda x: get_expected_info(x, level))

In [48]:
def calculate_accuracy_report(df):
    """Calculates accuracy breakdown for Part B."""
    print("\n" + "="*60)
    print("      SINGAPORE MATH CLASSIFIER: ACCURACY REPORT")
    print("="*60)
    
    levels = ['strand', 'subStrand', 'topic', 'learningOutcome', 'loId']
    
    for level in levels:
        # Check if both columns exist
        pred_col = level
        expected_col = f'expected_{level}' if level != 'loId' else 'expected_loId'
        
        if pred_col in df.columns and expected_col in df.columns:
            # Calculate accuracy
            correct = (df[pred_col] == df[expected_col]).mean() * 100
            print(f"Accuracy [{level.upper():<15}]: {correct:>6.2f}%")
        else:
            print(f"Skipping [{level.upper()}]: Columns not found")
            
    print("="*60 + "\n")

def display_mismatches(df):
    """Shows comparison view for mismatches."""
    mismatches = df[df['loId'] != df['expected_loId']]
    
    print(f"--- Comparison View: Mismatches Found ({len(mismatches)}) ---")
    for i, row in mismatches.iterrows():
        print(f"Q: {row['question'][:80]}...")
        print(f"   - EXPECTED: {row['expected_loId']}")
        print(f"   - ENGINE  : {row['loId']}  <-- MISMATCH")
        print("-" * 30)

In [51]:
def run_cli():
    while True:
        print("\n--- MATH CLASSIFIER ENGINE MENU ---")
        print("1. Run all 67 API questions (Batch Test)")
        print("2. Test custom question (Manual Input)")
        print("3. Exit")
        
        choice = input("\nEnter your choice (1/2/3): ")
        
        if choice == '1':
            print("\nRunning Batch Processing...")
            # This triggers your loop logic from previous steps
            # Ensure final_df is generated here
            calculate_accuracy_report(final_df)
            
            show_details = input("Show comparison for mismatches? (y/n): ")
            if show_details.lower() == 'y':
                display_mismatches(final_df)
                
        elif choice == '2':
            custom_q = input("\nEnter math question: ")
            print("\nClassifying...")
            result = classification_engine(custom_q)
            print(json.dumps(result, indent=4))
            
        elif choice == '3':
            print("Exiting engine. Goodbye!")
            break
        else:
            print("Invalid choice, try again.")


run_cli()


--- MATH CLASSIFIER ENGINE MENU ---
1. Run all 67 API questions (Batch Test)
2. Test custom question (Manual Input)
3. Exit

Running Batch Processing...

      SINGAPORE MATH CLASSIFIER: ACCURACY REPORT
Accuracy [STRAND         ]:  97.01%
Accuracy [SUBSTRAND      ]:  97.01%
Accuracy [TOPIC          ]:  74.63%
Accuracy [LEARNINGOUTCOME]:  56.72%
Accuracy [LOID           ]:  55.22%


--- MATH CLASSIFIER ENGINE MENU ---
1. Run all 67 API questions (Batch Test)
2. Test custom question (Manual Input)
3. Exit

Running Batch Processing...

      SINGAPORE MATH CLASSIFIER: ACCURACY REPORT
Accuracy [STRAND         ]:  97.01%
Accuracy [SUBSTRAND      ]:  97.01%
Accuracy [TOPIC          ]:  74.63%
Accuracy [LEARNINGOUTCOME]:  56.72%
Accuracy [LOID           ]:  55.22%

--- Comparison View: Mismatches Found (30) ---
Q: What is the remainder when 3079 is divided by 8?...
   - EXPECTED: P3:WHOLE NUMBERS:1.3.3
   - ENGINE  : P4:WHOLE NUMBERS:1.3.2  <-- MISMATCH
------------------------------
Q: 5 x 